<a href="https://colab.research.google.com/github/hfelizzola/Investigaciones-de-Operaciones-I/blob/main/programacion-lineal/03_problema_produccion_inventario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Investigación de Operaciones I: Planificación de Producción e Inventarios Multiperiodo

## Enunciado del Problema

**(Taha, 2010)** Se contrató a una compañía para que manufacturara dos productos, A y B, durante los meses de junio, julio y agosto. La capacidad de producción total (expresada en horas) varía mensualmente. La siguiente tabla proporciona los datos básicos de la situación:

| | Junio | Julio | Agosto |
| :--- | :---: | :---: | :---: |
| **Demanda de A (unidades)** | 500 | 5000 | 750 |
| **Demanda de B (unidades)** | 1000 | 1200 | 1200 |
| **Capacidad (h)** | 5000 | 5000 | 5000 |

**Nota**: *Se modificaron las capacidades respecto al ejercicio original para lograr una solución factible.*

Las tasas de producción por hora son 0.75 y 1 para los productos A y B, respectivamente. Se debe satisfacer toda la demanda; sin embargo, la de un mes posterior se puede satisfacer con la producción de uno anterior. Para cualquiera de los productos A y B guardados de un mes al siguiente, los costos de retención son de $\$0.90$ y $\$0.75$ por unidad, respectivamente. Los costos de producción unitarios de los dos productos, A y B, son de $\$30$ y $\$28$, respectivamente. Desarrolle un modelo de PL para determinar el programa de producción óptimo para los dos productos.

---

### Modelo Matemático

Este es un problema clásico de **flujo de inventarios multiperiodo**.

#### 1. Conjuntos e Índices
* $P$: Conjunto de productos, indexado por $i$.
    $$P = \{A, B\}$$
* $T$: Conjunto de periodos (meses), indexado por $t$.
    $$T = \{\text{Jun}, \text{Jul}, \text{Ago}\}$$

#### 2. Parámetros
* $D_{it}$: Demanda del producto $i$ en el mes $t$ (unidades).
* $K_t$: Capacidad de producción disponible en el mes $t$ (horas).
* $r_i$: Tasa de producción del producto $i$ (unidades por hora). Note que el tiempo requerido por unidad es $1/r_i$.
* $CP_i$: Costo de producción unitario del producto $i$ (\$).
* $CH_i$: Costo de almacenamiento (holding) unitario por mes del producto $i$ (\$).

#### 3. Variables de Decisión
* $x_{it} \geq 0$: Cantidad de unidades del producto $i$ a fabricar en el mes $t$.
* $I_{it} \geq 0$: Cantidad de unidades del producto $i$ en inventario al **final** del mes $t$.

#### 4. Función Objetivo
Minimizar el Costo Total (Producción + Inventario):
$$\text{Min } Z = \sum_{t \in T} \sum_{i \in P} (CP_i \cdot x_{it} + CH_i \cdot I_{it})$$

#### 5. Restricciones

**R1. Balance de Inventario (Ecuación de flujo):**
El inventario inicial más lo producido debe ser igual a la demanda más el inventario final. Asumimos un inventario inicial de cero ($I_{i,0} = 0$).
$$I_{it} = I_{i,t-1} + x_{it} - D_{it}  \quad \forall i \in P, \forall t \in T$$

**R2. Capacidad de Producción (Horas):**
La suma de las horas utilizadas por ambos productos no debe exceder la capacidad mensual.
$$\sum_{i \in P} \frac{1}{r_i} x_{it} \leq K_t \quad \forall t \in T$$

**R3. No Negatividad:**
$$x_{it}, I_{it} \geq 0 \quad \forall i \in P, \forall t \in T$$

**Paso 1: Instalación de la librería**

GAMSPY es la interfaz matemática de GAMS para Python. Primero, debemos instalarla en nuestro entorno de Colab.

In [ ]:
# Instalar GAMSPY en Google Colab
!pip install gamspy -q

**Paso 2: Inicializar el contenedor y definir los conjuntos**

En GAMSPY, todo modelo vive dentro de un `Container`. Luego, definiremos nuestros índices principales (conjuntos): los productos y los meses.

In [ ]:
import gamspy as gp

# 1. Crear el contenedor principal
m = gp.Container()

# 2. Definir los Conjuntos (Sets)
# El conjunto 'i' representa los productos
i = gp.Set(m, name="i", records=["A", "B"], description="Productos a manufacturar")

# El conjunto 't' representa los periodos de tiempo
t = gp.Set(m, name="t", records=["jun", "jul", "ago"], description="Meses de planificación")

print("Conjuntos creados exitosamente.")

Conjuntos creados exitosamente.


**Paso 3: Definir los Parámetros (Datos del problema)**

Para mantenerlo simple, usaremos listas de tuplas de Python para ingresar nuestros datos. Una tupla `("A", "Jun", 500)` significa que para el producto A en el mes de Junio, el valor es 500.

In [ ]:
# Demanda D(i, t)
datos_demanda = [
    ("A", "jun", 500), ("A", "jul", 5000), ("A", "ago", 750),
    ("B", "jun", 1000), ("B", "jul", 1200), ("B", "ago", 1200)
]
D = gp.Parameter(m, name="D", domain=[i, t], records=datos_demanda, description="Demanda mensual")

# Capacidad K(t) en horas
datos_capacidad = [
    ("jun", 5000), ("jul", 5000), ("ago", 5000)
]
K = gp.Parameter(m, name="K", domain=[t], records=datos_capacidad, description="Capacidad mensual en horas")

# Tasas de producción r(i) en unidades/hora
datos_tasas = [("A", 0.75), ("B", 1)]
r = gp.Parameter(m, name="r", domain=[i], records=datos_tasas, description="Tasa de producción")

# Costo de Producción CP(i)
datos_cp = [("A", 30), ("B", 28)]
CP = gp.Parameter(m, name="CP", domain=[i], records=datos_cp, description="Costo unitario de producción")

# Costo de Inventario CH(i)
datos_ch = [("A", 0.90), ("B", 0.75)]
CH = gp.Parameter(m, name="CH", domain=[i], records=datos_ch, description="Costo unitario de inventario por mes")

print("Parámetros cargados.")

Parámetros cargados.


**Paso 4: Variables de Decisión**

Definiremos las variables de producción ($x_{it}$) y de inventario ($I_{it}$). Ambas deben ser mayores o iguales a cero (tipo `Positive`). También necesitamos una variable libre para almacenar el valor de la función objetivo ($Z$).

In [ ]:
# Variable de Producción: Cantidad a producir del producto i en el mes t
x = gp.Variable(m, name="x", domain=[i, t], type="Positive", description="Unidades producidas")

# Variable de Inventario: Cantidad guardada del producto i al final del mes t
inv = gp.Variable(m, name="inv", domain=[i, t], type="Positive", description="Unidades en inventario")

# Variable Objetivo: Costo Total
Z = gp.Variable(m, name="Z", type="Free", description="Costo Total")

**Paso 5: Ecuaciones**

Aquí está la magia del modelado algebraico.
1. **Balance de inventario:** El inventario del mes anterior (`t.lag(1)`) más lo producido hoy, debe ser igual a la demanda de hoy más el inventario que guardo para el mes siguiente. *Nota: GAMSPY es inteligente, si estamos en el primer mes, `t.lag(1)` automáticamente se asume como cero, asumiendo que no hay inventario inicial.*
2. **Capacidad:** El tiempo total gastado produciendo (cantidad / tasa) no puede superar la capacidad del mes.

In [ ]:
# 1. Ecuación de Balance de Inventario
balance_inv = gp.Equation(m, name="balance_inv", domain=[i, t], description="Balance de flujo de inventarios")

# La sintaxis t.lag(1) hace referencia al mes anterior (t-1)
balance_inv[i, t] = inv[i, t] == inv[i, t.lag(1)] + x[i, t] - D[i, t]

# 2. Ecuación de Capacidad de Producción
capacidad_prod = gp.Equation(m, name="capacidad_prod", domain=[t], description="Limite de horas de produccion")

# Sumamos sobre 'i' el tiempo requerido para cada producto: (1 / r[i]) * x[i, t]
capacidad_prod[t] = gp.Sum(i, (1 / r[i]) * x[i, t]) <= K[t]

# 3. Función Objetivo (Minimizar Costos)
funcion_objetivo = gp.Equation(m, name="funcion_objetivo", description="Calculo del costo total")

# Sumatoria de los costos de produccion y los costos de mantener inventario
funcion_objetivo[...] = Z == gp.Sum((i, t), CP[i] * x[i, t] + CH[i] * inv[i, t])

print("Ecuaciones definidas algebraicamente.")

Ecuaciones definidas algebraicamente.


**Paso 6: Construcción del Modelo y Solución**

Finalmente, empaquetamos nuestras ecuaciones en un modelo matemático de tipo Programación Lineal (LP) y le pedimos al motor que encuentre la solución óptima (minimizar).

In [ ]:
# Crear el modelo incluyendo todas las ecuaciones definidas en el contenedor 'm'
modelo_produccion = gp.Model(
    m,
    name="Planificacion_Taha",
    equations=m.getEquations(),
    problem="LP",         # Linear Programming
    sense=gp.Sense.MIN,   # Minimizar
    objective=Z           # Variable a optimizar
)

# Resolver el modelo
modelo_produccion.solve(options=gp.Options(equation_listing_limit=100))


,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,284635.0,10,13,LP,CPLEX,0.0


In [ ]:
# --- Mostrar Resultados de forma amigable ---
print("\n" + "="*50)
print(f"ESTADO DE LA SOLUCIÓN: {modelo_produccion.status}")
print(f"COSTO TOTAL MÍNIMO (Z): ${Z.records.level[0]:,.2f}")
print("="*50 + "\n")

print("PLAN DE PRODUCCIÓN (x_it):")
# Filtramos para mostrar solo los valores mayores a cero (level > 0)
print(x.records[x.records['level'] > 0][['i', 't', 'level']].to_string(index=False))

print("\nPLAN DE INVENTARIOS (I_it):")
# Filtramos inventarios mayores a cero
df_inv = inv.records[inv.records['level'] > 0][['i', 't', 'level']]
if not df_inv.empty:
    print(df_inv.to_string(index=False))
else:
    print("No se guarda inventario en ningún periodo.")


ESTADO DE LA SOLUCIÓN: ModelStatus.OptimalGlobal
COSTO TOTAL MÍNIMO (Z): $284,635.00

PLAN DE PRODUCCIÓN (x_it):
i   t  level
A jun 2650.0
A jul 2850.0
A ago  750.0
B jun 1000.0
B jul 1200.0
B ago 1200.0

PLAN DE INVENTARIOS (I_it):
i   t  level
A jun 2150.0


### **Análisis de la Solución Óptima**

Una vez que el motor de optimización encuentra la solución global (`ModelStatus.OptimalGlobal`), es fundamental interpretar qué significan estos números en el contexto de la planta de manufactura. El costo mínimo garantizado para operar durante estos tres meses es de **$284,635.00**.

Al analizar el plan de producción ($x_{it}$) y el plan de inventarios ($I_{it}$), podemos extraer las siguientes conclusiones estratégicas:

* **Anticipación al "Pico" de Demanda (Producto A):** El comportamiento más interesante del modelo ocurre con el producto A. En el mes de julio, la demanda se dispara a 5000 unidades. Dado que es matemáticamente imposible fabricar toda esa cantidad en un solo mes sin violar la restricción de capacidad de horas, el modelo decide "adelantar" trabajo. En junio, fabrica 2650 unidades, cubre la demanda de 500 y guarda el sobrante ($I_{A,\text{jun}} = 2150$) en inventario. En julio, suma ese inventario a las 2850 unidades que fabrica para cumplir exactamente con la meta de 5000.
* **Producción "Justo a Tiempo" (Producto B):** A diferencia del producto A, la producción del producto B calca exactamente su demanda mensual (1000 en junio, 1200 en julio y 1200 en agosto). Como la capacidad en horas lo permite, el modelo es inteligente y decide no almacenar este producto en absoluto, ahorrándose por completo el costo de retención de **$0.75** por unidad.
* **Vaciado del Sistema:** Como es típico en los modelos determinísticos de horizonte finito (sin políticas de inventario de seguridad), al finalizar el mes de agosto el inventario de ambos productos cae a cero. El optimizador no fabricará ni una unidad extra que genere costos si no hay una demanda futura que justifique ese gasto.

---